# Component B, Question 2: Consensus and Swarm Intelligence

1. **Average consensus:** the robots agree on a rendezvous point using only neighbour communication.
2. **Particle Swarm Optimisation (PSO):** the robots act as particles that cooperatively search for the signal source.
3. **Emergent behaviour and convergence evaluation.**

In [ ]:
import sys
sys.path.append("..")
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import swarm_data as sd

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110})

## 1. Average consensus

Each robot *i* holds an estimate *xᵢ* (a 2-D point), initialised to its own
start position. Every round (0.25 s), it sends *xᵢ* to its ring neighbours and
updates:

$$x_i \leftarrow x_i + \varepsilon \sum_{j \in N(i)} (x_j - x_i), \qquad \varepsilon = 0.3$$

In matrix form this is *x ← (I − εL)x*, where *L* is the graph Laplacian. The
sum of all estimates never changes, so if the estimates agree, they agree on
the **true average**. For a 4-robot ring, the Laplacian's eigenvalues are
{0, 2, 2, 4}. With ε = 0.3, the update matrix has eigenvalues {1, 0.4, 0.4, −0.2}.
Disagreement should therefore shrink by a factor of **0.4 per round**
(stability requires ε < 1/max degree = 0.5).

In [ ]:
RUN = "distributed_s1"
estimates = []
for i in range(sd.N_ROBOTS):
    d = sd.robot_log(RUN, i)
    c = d[d.phase == "consensus"].groupby("consensus_round").first()
    estimates.append(c[["state_x", "state_y"]].values)
estimates = np.array(estimates)                 # robot x round x 2
start_mean = estimates[:, 0, :].mean(axis=0)    # average of actual starts

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rounds = np.arange(estimates.shape[1])
for i in range(sd.N_ROBOTS):
    axes[0].plot(rounds, estimates[i, :, 0], "o-", ms=3,
                 color=sd.ROBOT_COLOURS[i], label=f"robot_{i}")
    axes[1].plot(rounds, estimates[i, :, 1], "o-", ms=3,
                 color=sd.ROBOT_COLOURS[i])
for ax, k, label in [(axes[0], 0, "x"), (axes[1], 1, "y")]:
    ax.axhline(start_mean[k], c="k", ls="--", lw=1,
               label="average of start positions")
    ax.set_xlabel("consensus round")
    ax.set_ylabel(f"estimate {label} (m)")
axes[0].legend(fontsize=7)
# Disagreement: largest distance of any robot's estimate from the group mean.
disagreement = np.max(np.linalg.norm(
    estimates - estimates.mean(axis=0, keepdims=True), axis=2), axis=0)
axes[2].semilogy(rounds, disagreement, "o-", ms=3, label="measured")
axes[2].semilogy(rounds, disagreement[0] * 0.4 ** rounds, "k--",
                 label="theory: x 0.4 per round")
axes[2].set_xlabel("consensus round")
axes[2].set_ylabel("disagreement between robots (m)")
axes[2].legend(fontsize=8)
axes[0].set_title("Estimates converge (x)")
axes[1].set_title("Estimates converge (y)")
axes[2].set_title("Convergence rate")
plt.tight_layout()
plt.savefig(OUT_DIR / "b2_consensus.png", bbox_inches="tight")
plt.show()

final = estimates[:, -1, :]
print("Average of the robots' actual start positions:", np.round(start_mean, 4))
print("Agreed point per robot:", np.round(final, 4).tolist())
first_below = int(np.argmax(disagreement < 0.01))
print(f"Disagreement below 1 cm after {first_below} rounds "
      f"({first_below * 0.25:.2f} s); final disagreement "
      f"{1000 * disagreement[-1]:.3f} mm")

**Interpretation.** All four estimates converge on the same point: the
average of the robots' actual start positions. That point is a few
millimetres from the nominal start average, because the robots settle
slightly off their nominal start spots after being placed. No robot ever knew
all four positions. Disagreement falls along a straight line on the log
plot, matching the predicted factor of 0.4 per round, until it reaches the
rounding precision of the logged values. The centralised alternative needs a
single round, because the coordinator sees every position. That speed is
exactly what makes the coordinator a single point of failure (Question 3).

## 2. Formation (using the consensus result)

In [ ]:
centre = np.array(final[0], dtype=float)
rows = []
for i in range(sd.N_ROBOTS):
    d = sd.robot_log(RUN, i)
    f = d[d.phase == "formation"].iloc[-1]
    slot = centre + 0.25 * np.array([math.cos(math.pi / 2 * i),
                                     math.sin(math.pi / 2 * i)])
    rows.append({"Robot": i, "Slot (x, y)": tuple(np.round(slot, 3)),
                 "Reached (x, y)": (round(f.x, 3), round(f.y, 3)),
                 "Error (cm)": round(100 * math.hypot(f.x - slot[0],
                                                      f.y - slot[1]), 1)})
pd.DataFrame(rows)

Each robot drives to its own slot on a circle around the *agreed* point, so a
shared decision becomes a coordinated spatial arrangement. The robots stop
within the 5 cm arrival tolerance. This is the starting configuration for
the search.

## 3. Particle Swarm Optimisation for the search

Each robot is a PSO particle whose position is its real position in the
arena. Every iteration (4 s), each robot:
1. reads the signal *f* at its position and updates its personal best (*pbest*) if *f* is higher
2. shares *pbest* and its value (with ring neighbours, or with the coordinator)
3. updates its **global best (gbest)**: the best value it has heard of
4. computes its velocity: *v ← w·v + c₁r₁(pbest − x) + c₂r₂(gbest − x)*, with *w* = 0.7, *c₁* = *c₂* = 1.5, and step length capped at 0.4 m
5. drives to *x + v* using the navigation policy, avoiding obstacles and other robots on the way

In the **distributed** version, *gbest* is really a *local* best: each robot
only hears its ring neighbours. Good information therefore spreads around the
ring over successive iterations instead of arriving instantly.

In [ ]:
X, Y = np.meshgrid(np.linspace(-1, 1, 200), np.linspace(-1, 1, 200))
field = np.exp(-((X - sd.SOURCE[0]) ** 2 + (Y - sd.SOURCE[1]) ** 2)
               / (2 * 0.5 ** 2))

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, e in zip(axes, ["distributed_s1", "centralised_s1"]):
    ax.contourf(X, Y, field, levels=15, cmap="YlOrBr", alpha=0.6)
    sd.draw_arena(ax)
    for i in range(sd.N_ROBOTS):
        p = sd.robot_log(e, i).query("phase == 'pso'")
        ax.plot(p.x, p.y, color=sd.ROBOT_COLOURS[i], lw=1.4,
                label=f"robot_{i}")
        ax.plot(p.x.iloc[0], p.y.iloc[0], "o", color=sd.ROBOT_COLOURS[i],
                mec="k")
        ax.plot(p.x.iloc[-1], p.y.iloc[-1], "X", color=sd.ROBOT_COLOURS[i],
                mec="k", ms=9)
    ax.set_title(f"PSO search: {e} (o = start, X = end)")
    ax.legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.savefig(OUT_DIR / "b2_pso_trajectories.png", bbox_inches="tight")
plt.show()

### Convergence

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
prog = sd.pso_progress("distributed_s1")
for i in range(sd.N_ROBOTS):
    r = prog[prog.robot == i]
    axes[0].plot(r.iteration, 100 * r.gbest_error, "-o", ms=3,
                 color=sd.ROBOT_COLOURS[i], label=f"robot_{i}")
axes[0].set_ylabel("gbest distance to source (cm)")
axes[0].set_title("Distributed, seed 1: each robot's best-known position")
axes[0].legend(fontsize=7)

for kind, colour in [("distributed", "tab:blue"), ("centralised", "tab:orange")]:
    dist_curves, spread_curves = [], []
    for run in sd.runs(kind):
        sup = sd.supervisor_log(run)
        sup = sup[sup.t > 26.25]
        xs = sup[[f"x{i}" for i in range(4)]].values
        ys = sup[[f"y{i}" for i in range(4)]].values
        it = (sup.t.values - 26.25) / 4
        dist = np.mean(np.hypot(xs - sd.SOURCE[0], ys - sd.SOURCE[1]), axis=1)
        spread = np.mean(np.hypot(xs - xs.mean(1, keepdims=True),
                                  ys - ys.mean(1, keepdims=True)), axis=1)
        axes[1].plot(it, 100 * dist, color=colour, lw=0.6, alpha=0.4)
        axes[2].plot(it, 100 * spread, color=colour, lw=0.6, alpha=0.4)
        n = min(len(dist), *(len(c) for c in dist_curves)) if dist_curves             else len(dist)
        dist_curves.append(dist)
        spread_curves.append(spread)
    n = min(len(c) for c in dist_curves)
    axes[1].plot(it[:n], 100 * np.mean([c[:n] for c in dist_curves], axis=0),
                 color=colour, lw=2.2, label=f"{kind} (mean of seeds)")
    axes[2].plot(it[:n], 100 * np.mean([c[:n] for c in spread_curves], axis=0),
                 color=colour, lw=2.2, label=f"{kind} (mean of seeds)")
axes[1].set_ylabel("mean robot distance to source (cm)")
axes[1].set_title("Where the robots actually are")
axes[2].set_ylabel("mean distance to swarm centre (cm)")
axes[2].set_title("Swarm spread")
for ax in axes[1:]:
    ax.legend(fontsize=8)
for ax in axes:
    ax.set_xlabel("PSO iteration")
plt.tight_layout()
plt.savefig(OUT_DIR / "b2_pso_convergence.png", bbox_inches="tight")
plt.show()

In [ ]:
convergence = sd.kind_table(["distributed", "centralised"],
                            ["first within 10 cm (iter)",
                             "all within 10 cm (iter)", "gbest error (cm)",
                             "robot distance (cm)", "contact events",
                             "closest gap (mm)"])
convergence.to_csv(OUT_DIR / "b2_convergence.csv")
convergence

**Interpretation: emergent behaviour.** No robot is told to move towards the
source or towards the others, yet the swarm **converges on the source**. The
mean robot distance falls from about 1 m to about 20 cm within 8–10
iterations, an emergent result of four copies of the same simple update rule.
Three further effects are visible:

- **Information diffusion in the ring.** In the distributed run (left), the robots' *gbest* curves do not drop together: robot 0 repeatedly lags one or two iterations behind, because a discovery reaches a neighbour after one iteration and the robot opposite after two. Across the three seeds, the first robot got within 10 cm *earlier* in distributed mode than in centralised mode, but the last robot learned of it about five iterations later (table). With the coordinator, everyone learns the best position in the same iteration.
- **Residual oscillation.** After converging, the robots hover 20–35 cm from the source, and the swarm's spread rises and falls. Particles keep swinging past the best point (PSO momentum), and they push each other away when they meet. The inertia weight falls from 0.9 to 0.4 over the search to damp this, which is why the swings shrink towards the end. With more iterations, or a faster-falling weight, the swarm would settle further.
- **Crowding.** The robots cannot all occupy the source at once, and collision avoidance keeps them apart. So the *actual* robot distance stays above the *gbest* error. The only body contacts in all twelve runs happened in a centralised run, where every robot was steering to the identical point at the same moment. In the ring, robots chase slightly different local bests and spread out naturally, an unplanned safety benefit of distributed decision-making.

**Convergence evaluation.** Both versions bring the best-known position to
about 6 cm from the true source, a difference smaller than the
seed-to-seed spread. The modes differ in *how* the knowledge spreads, not in
the final answer.